In [3]:
import pandas as pd
import numpy as np
import pandas_ta as ta
import yfinance as yf
import requests # requests sert à envoyer une requête vers un site web pour récupérer son contenu.
from io import StringIO

In [4]:
import sys
print(sys.executable)


C:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Scripts\python.exe


In [5]:
end_date = pd.to_datetime('2023-01-01')
start_date = pd.to_datetime(end_date) - pd.DateOffset(years=5)
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

In [6]:
def get_sp500_tickers(url):
    """
    Get the list of S&P 500 tickers from Wikipedia.
    """
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        sp500_table = pd.read_html(StringIO(response.text))[0]
    return sp500_table['Symbol'].str.replace(".", "-").tolist()    

def download_data(tickers, start_date, end_date):
    """
    Download historical stock data for the given tickers and date range.
    """
    df = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=True, threads=True)
    return df



Un header, c’est une information envoyée avec la requête HTTP.

Ici, on dit au site :

Je suis un navigateur web normal

Sans ça, Wikipedia peut répondre :

403 Forbidden

Parce qu’il pense que la requête vient d’un robot ou d’un script automatique.<br>
Les statuts importants :<br>

<Response [200]>  = OK, page récupérée<br>
<Response [403]>  = Forbidden, accès bloqué<br>
<Response [404]>  = page introuvable<br>
<Response [500]>  = erreur serveur<br>

Tu utilises StringIO quand tu as déjà du texte en mémoire, mais qu’une fonction veut lire ce texte comme si c’était un fichier.

In [18]:
tickers = get_sp500_tickers(url)
df = download_data(tickers, start_date, end_date)

[**********            20%                       ]  103 of 503 completed$GEV: possibly delisted; no price data found  (1d 2018-01-01 00:00:00 -> 2023-01-01 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1672549200")
[***********           23%                       ]  116 of 503 completed$FDXF: possibly delisted; no price data found  (1d 2018-01-01 00:00:00 -> 2023-01-01 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1672549200")
[*****************     36%                       ]  182 of 503 completed$VLTO: possibly delisted; no price data found  (1d 2018-01-01 00:00:00 -> 2023-01-01 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1672549200")
[**********************57%**                     ]  286 of 503 completed$SNDK: possibly delisted; no price data found  (1d 2018-01-01 00:00:00 -> 2023-01-01 00:00:00) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 16

In [ ]:
df = df.stack(level=0)
df.columns.name = None
df.drop(columns=["Adj Close"], inplace=True)
df.rename(columns={
    "Close": "close",
    "High": "high",
    "Low": "low",
    "Open": "open",
    "Volume": "volume"
}, inplace=True)

C:\Users\bilal\AppData\Local\Temp\ipykernel_26016\794733328.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(level=0)


In [56]:
print(df.isnull().sum())
print(df.index.names)

Price
open      0
high      0
low       0
close     0
volume    0
dtype: int64
['Date', 'Ticker']


In [ ]:
class FeatureEngineering:
    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe.reset_index().sort_values(["Ticker", "Date"]).copy()

    def daily_return(self):
        close_yesterday = self.df.groupby("Ticker")["close"].shift(1)
        self.df["daily_return"] = (
            self.df["close"] - close_yesterday
        ) / close_yesterday
        return self.df

    def monthly_return(self):
        close_21day = self.df.groupby("Ticker")["close"].shift(21)
        self.df["monthly_return"] = (
            self.df["close"] - close_21day
        ) / close_21day
        return self.df

    def rsi(self):
        self.df["rsi"] = self.df.groupby("Ticker")["close"].transform(
            lambda x: ta.rsi(close=x, length=20)
        )
        return self.df

    def dollar_volume(self):
        self.df["dollar_volume"] = self.df["volume"] * self.df["close"]
        return self.df